                # MIPROv2

                Jointly propose instructions and demonstrations, then use validation feedback to search their combinations.

                **Use it when:** Both prompt wording and examples may matter and you can spend more compile calls for a joint search.

                **What compilation changes:** Instruction plus up to two bootstrapped and two labeled demonstrations.

                Important configuration in this benchmark:

                - `auto='light'` search budget
- seed 42 with minibatched validation
- Sol proposes prompts; Luna runs the task

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'miprov2'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='miprov2'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.MIPROv2(
    metric=exact_match, prompt_model=reflection_lm, task_model=task_lm,
    auto='light', max_bootstrapped_demos=2, max_labeled_demos=2, seed=42,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, valset=valset, minibatch=True,
    requires_permission_to_run=False,
)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

MIPROv2 — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 95.0%
uplift: +45.0 points vs Luna reference
optimization: $0.5072 and 7.6min
inference latency: mean 1.67s; p95 2.87s
reload checks: prompt=True; model=None; predictions=3/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/miprov2.json
- canonical prompt: chapter06/results/final_prompts/miprov2.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Inspect instruction and demos together: MIPROv2 optimizes their combination, so attributing the result to either one alone would be misleading.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (1724 characters):
You are the final forensic reviewer in a high-stakes technical publishing audit. A mistaken verdict could either allow undisclosed AI-rewritten documentation into a safety-critical release or falsely accuse a human author, so assess each passage carefully and impartially.

Determine whether the supplied passage is AI-generated or AI-paraphrased (`True`) versus human-written (`False`). Because paired passages may contain nearly identical technical facts, judge the **writing style rather than the topic, correctness, or sophistication of the content**.

Look for clusters of evidence:

- **AI tendencies:** conspicuously polished transitions; elevated, archaic, or needlessly formal diction; excessive syntactic smoothing; balanced or uniformly fluent sentence structures; formal punctuation; conversion of concise steps or lists into flowing prose; awkwardly ornate wording; generic broad claims; mild embellishment or reinterpreta

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.